# 11 — What the corrupted labels do to a deployed model

One experiment, four reviewer questions:

1. **Common-test evaluation** — train on original vs corrected labels, evaluate
   BOTH on the same corrected-label test flows. Separates "the test labels
   changed" from "the model learned something different".
2. **Prediction disagreement** — with stored predictions: how often do the two
   models disagree, and what does the original-label model call the
   benign→attack flows?
3. **Per-class table** — which families actually benefit from correction.
4. **Paired bootstrap CIs** — 1,000 resamples over the ~481k common test flows,
   the statistically meaningful CI for this design (seeds only measure model
   randomness; flows measure the estimate).

Plus the **Attempted-as-third-class** policy, completing H3.

All on the matched subset (train and test share the feature matrix; only the
training labels differ). Runtime ≈ 1–2 h.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)
import pandas as pd, numpy as np, json, time

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report
from xgboost import XGBClassifier

matched = pd.read_parquet(os.path.join(C.INTERIM, 'matched.parquet'))
base, rep = H.clean_features(matched)
print(len(base), 'clean matched flows')

FEATS = [c for c in base.columns
         if c not in ('label_original', 'label_improved', 'attempted', 'label',
                      'timestamp', 'src_ip', 'dst_ip', 'flow_id', 'day')
         and pd.api.types.is_numeric_dtype(base[c])]
print(len(FEATS), 'features')

# ONE fixed split, stratified on the corrected binary label; identical for all
SPLIT_SEED = 11
tr_idx, te_idx = train_test_split(
    base.index, test_size=0.30, random_state=SPLIT_SEED,
    stratify=H.binarise(base['label_improved']))
TR, TE = base.loc[tr_idx], base.loc[te_idx]
print(f'train {len(TR):,}  test {len(TE):,}')

def make_model(name, seed=11):
    if name == 'logreg':
        return LogisticRegression(max_iter=1000, n_jobs=-1, random_state=seed)
    if name == 'random_forest':
        return RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
    if name == 'xgboost':
        return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.3,
                             tree_method='hist', n_jobs=-1, random_state=seed,
                             eval_metric='logloss')
    raise ValueError(name)

Mounted at /content/drive
1604990 clean matched flows
72 features
train 1,123,493  test 481,497


In [2]:
# Train each model once per label source; store family-level predictions.
PRED_PATH = os.path.join(C.INTERIM, 'predictions_common_test.parquet')
MODELS = ['logreg', 'random_forest', 'xgboost']

if os.path.exists(PRED_PATH):
    preds = pd.read_parquet(PRED_PATH)
    print('predictions cached:', preds.shape)
else:
    Xtr, Xte = TR[FEATS].values, TE[FEATS].values
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

    preds = pd.DataFrame(index=TE.index)
    preds['y_family_original'] = H.coarse_class(TE['label_original']).values
    preds['y_family_improved'] = H.coarse_class(TE['label_improved']).values

    from sklearn.preprocessing import LabelEncoder
    for src in ['label_original', 'label_improved']:
        ytr = H.coarse_class(TR[src]).values
        for m in MODELS:
            t0 = time.time()
            X1 = Xtr_s if m == 'logreg' else Xtr
            X2 = Xte_s if m == 'logreg' else Xte
            if m == 'xgboost':   # xgboost needs integer classes
                le = LabelEncoder().fit(ytr)
                mdl = make_model(m).fit(X1, le.transform(ytr))
                p = le.inverse_transform(mdl.predict(X2))
            else:
                mdl = make_model(m).fit(X1, ytr)
                p = mdl.predict(X2)
            col = f'pred_{m}_{src.replace("label_", "")}'
            preds[col] = p
            print(f'{col:45s} {time.time()-t0:6.1f}s')
    preds.to_parquet(PRED_PATH)
    print('stored', PRED_PATH)

predictions cached: (481497, 8)


In [3]:
# 1) COMMON-TEST evaluation: both models scored on corrected-label test
rows = []
for m in MODELS:
    for trained_on in ['original', 'improved']:
        p = preds[f'pred_{m}_{trained_on}']
        for scored_on in ['improved', 'original']:
            y = preds[f'y_family_{scored_on}']
            yb, pb = (y != 'BENIGN').astype(int), (p != 'BENIGN').astype(int)
            rows.append({
                'model': m, 'trained_on': trained_on, 'scored_against': scored_on,
                'macro_f1_family': round(f1_score(y, p, average='macro',
                                                  zero_division=0), 4),
                'binary_attack_recall': round(recall_score(yb, pb), 4),
                'binary_attack_precision': round(precision_score(yb, pb,
                                                 zero_division=0), 4)})
ct = pd.DataFrame(rows)
H.save_table(ct, 't11_common_test.csv')
print('Key comparison - same corrected-label test flows, different training labels:')
ct[ct['scored_against'] == 'improved'].pivot_table(
    index='model', columns='trained_on',
    values=['macro_f1_family', 'binary_attack_recall'])

saved /content/drive/MyDrive/research/ids-label-correction/results/t11_common_test.csv (12, 6)
Key comparison - same corrected-label test flows, different training labels:


binary_attack_recall          macro_f1_family         
trained_on                improved original        improved original
model                                                               
logreg                      0.9885   0.8805          0.7526   0.6525
random_forest               0.9998   0.8151          0.9935   0.7525
xgboost                     0.9998   0.8150          0.9688   0.7477

In [4]:
# 2) PREDICTION DISAGREEMENT + the 65,001 benign->attack flows in the test set
rows = []
for m in MODELS:
    po = preds[f'pred_{m}_original']
    pi = preds[f'pred_{m}_improved']
    dis = (po != pi)

    # flows the correction relabelled benign->attack, falling in the test 30%
    b2a = ((preds['y_family_original'] == 'BENIGN') &
           (preds['y_family_improved'] != 'BENIGN'))
    caught_o = (po[b2a] != 'BENIGN').mean()
    caught_i = (pi[b2a] != 'BENIGN').mean()
    rows.append({'model': m,
                 'pred_disagreement_pct': round(100 * dis.mean(), 3),
                 'b2a_flows_in_test': int(b2a.sum()),
                 'orig_model_flags_pct': round(100 * caught_o, 2),
                 'corr_model_flags_pct': round(100 * caught_i, 2)})
pdis = pd.DataFrame(rows)
H.save_table(pdis, 't11_pred_disagreement.csv')
print('Of the benign->attack corrected flows in the test partition, the share')
print('each model flags as any attack:')
pdis

saved /content/drive/MyDrive/research/ids-label-correction/results/t11_pred_disagreement.csv (3, 5)
Of the benign->attack corrected flows in the test partition, the share
each model flags as any attack:


,model,pred_disagreement_pct,b2a_flows_in_test,orig_model_flags_pct,corr_model_flags_pct
0,logreg,2.792,19493,38.53,99.09
1,random_forest,4.047,19493,0.06,99.95
2,xgboost,4.063,19493,0.00,99.97


In [5]:
# 3) PER-CLASS table on the common corrected-label test
y = preds['y_family_improved']
rows = []
for fam in sorted(y.unique()):
    mask = (y == fam)
    r = {'family': fam, 'n_test': int(mask.sum())}
    for m in MODELS:
        for src in ['original', 'improved']:
            p = preds[f'pred_{m}_{src}']
            r[f'{m}_{src}_recall'] = round(float((p[mask] == fam).mean()), 4)
    rows.append(r)
pc = pd.DataFrame(rows).sort_values('n_test', ascending=False)
H.save_table(pc, 't11_perclass.csv')
pc

saved /content/drive/MyDrive/research/ids-label-correction/results/t11_perclass.csv (9, 8)


,family,n_test,logreg_original_recall,logreg_improved_recall,random_forest_original_recall,random_forest_improved_recall,xgboost_original_recall,xgboost_improved_recall
0,BENIGN,376030,0.9981,0.9968,1.0000,1.0000,0.9999,0.9999
7,PortScan,47513,0.9959,0.9988,0.9998,1.0000,0.9998,0.9999
4,DoS,31872,0.9836,0.9682,0.9952,0.9997,0.9951,0.9996
6,Infiltration,17047,0.0002,0.5545,0.0004,0.9989,0.0004,0.9995
3,DDoS,8546,0.7371,0.9982,0.7378,1.0000,0.7377,0.9995
2,BruteForce,364,0.9835,0.9835,1.0000,1.0000,0.9973,0.9890
8,WebAttack,63,0.2222,0.0952,0.9206,0.9365,0.9206,0.9683
1,Bot,59,0.0000,0.0169,0.0000,1.0000,0.0000,1.0000
5,Heartbleed,3,0.6667,1.0000,1.0000,1.0000,1.0000,0.6667


In [6]:
# 4) PAIRED BOOTSTRAP over test flows (1,000 resamples) for the deltas
rng = np.random.default_rng(7)
N = len(preds); B = 1000
y_i = preds['y_family_improved'].values
res = []
for m in MODELS:
    po = preds[f'pred_{m}_original'].values
    pi = preds[f'pred_{m}_improved'].values
    d_macro, d_rec = np.empty(B), np.empty(B)
    yb = (y_i != 'BENIGN').astype(int)
    pob, pib = (po != 'BENIGN').astype(int), (pi != 'BENIGN').astype(int)
    for b in range(B):
        idx = rng.integers(0, N, N)
        d_macro[b] = (f1_score(y_i[idx], pi[idx], average='macro', zero_division=0)
                      - f1_score(y_i[idx], po[idx], average='macro', zero_division=0))
        d_rec[b] = recall_score(yb[idx], pib[idx]) - recall_score(yb[idx], pob[idx])
    for name, d in [('macro_f1_family', d_macro), ('binary_attack_recall', d_rec)]:
        res.append({'model': m, 'metric': name,
                    'delta_mean': round(float(d.mean()), 4),
                    'ci95_low': round(float(np.percentile(d, 2.5)), 4),
                    'ci95_high': round(float(np.percentile(d, 97.5)), 4)})
    print(m, 'done')
boot = pd.DataFrame(res)
H.save_table(boot, 't11_bootstrap_ci.csv')
boot

logreg done
random_forest done
xgboost done
saved /content/drive/MyDrive/research/ids-label-correction/results/t11_bootstrap_ci.csv (6, 5)


,model,metric,delta_mean,ci95_low,ci95_high
0,logreg,macro_f1_family,0.1036,0.0651,0.1930
1,logreg,binary_attack_recall,0.1081,0.1062,0.1100
2,random_forest,macro_f1_family,0.2421,0.2372,0.2712
3,random_forest,binary_attack_recall,0.1846,0.1824,0.1870
4,xgboost,macro_f1_family,0.2178,0.1303,0.2752
5,xgboost,binary_attack_recall,0.1847,0.1822,0.1871


In [7]:
# 5) H3 completion: Attempted as a SEPARATE third class (improved, full corpus)
impr = pd.read_parquet(os.path.join(C.INTERIM, 'improved.parquet'))
impr, _ = H.clean_features(impr)
att = H.is_attempted(impr['label'])
fam = H.coarse_class(impr['label'])
impr['y3'] = np.where(att, fam + '-Attempted', fam)

F3 = [c for c in impr.columns
      if c not in ('label', 'attempted', 'timestamp', 'src_ip', 'dst_ip',
                   'flow_id', 'day', 'y3')
      and pd.api.types.is_numeric_dtype(impr[c])]
rows = []
for seed in C.SEEDS:
    i_tr, i_te = train_test_split(impr.index, test_size=0.30, random_state=seed,
                                  stratify=(impr['y3'] != 'BENIGN'))
    Xtr, Xte = impr.loc[i_tr, F3].values, impr.loc[i_te, F3].values
    ytr, yte = impr.loc[i_tr, 'y3'].values, impr.loc[i_te, 'y3'].values
    yte_fam = pd.Series(yte).str.replace('-Attempted', '', regex=False).values
    for m in MODELS:
        if m == 'logreg':
            sc = StandardScaler().fit(Xtr)
            X1, X2 = sc.transform(Xtr), sc.transform(Xte)
        else:
            X1, X2 = Xtr, Xte
        if m == 'xgboost':
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder().fit(ytr)
            mdl = make_model(m, seed).fit(X1, le.transform(ytr))
            p3 = pd.Series(le.inverse_transform(mdl.predict(X2)))
        else:
            mdl = make_model(m, seed).fit(X1, ytr)
            p3 = pd.Series(mdl.predict(X2))
        p_fam = p3.str.replace('-Attempted', '', regex=False).values
        rows.append({'seed': seed, 'model': m,
                     'macro_f1_17class': round(f1_score(yte, p3, average='macro',
                                               zero_division=0), 4),
                     'macro_f1_collapsed_9class': round(
                         f1_score(yte_fam, p_fam, average='macro',
                                  zero_division=0), 4)})
        print(seed, m, rows[-1]['macro_f1_collapsed_9class'])
h3c = pd.DataFrame(rows)
H.save_table(h3c, 't11_attempted_third_class.csv')
print('\nCollapsed 9-family macro-F1 is directly comparable to the as_attack')
print('policy rows in runs.csv - the third-class policy completes Table H3.')
h3c.groupby('model')[['macro_f1_17class', 'macro_f1_collapsed_9class']].agg(['mean', 'std']).round(4)

11 logreg 0.8165
11 random_forest 0.9951
11 xgboost 0.997
23 logreg 0.8464
23 random_forest 0.9953
23 xgboost 0.9698
37 logreg 0.7996
37 random_forest 0.9955
37 xgboost 0.9979
51 logreg 0.8124
51 random_forest 0.9955
51 xgboost 0.9602
73 logreg 0.825
73 random_forest 0.9955
73 xgboost 0.8162
saved /content/drive/MyDrive/research/ids-label-correction/results/t11_attempted_third_class.csv (15, 4)

Collapsed 9-family macro-F1 is directly comparable to the as_attack
policy rows in runs.csv - the third-class policy completes Table H3.


macro_f1_17class         macro_f1_collapsed_9class        
                          mean     std                      mean     std
model                                                                   
logreg                  0.7154  0.0126                    0.8200  0.0174
random_forest           0.9847  0.0049                    0.9954  0.0002
xgboost                 0.9005  0.1700                    0.9482  0.0756